# 🚀 Data Platform v2.0 – Test
**PyCharm Pro → Jupyter**

| Kanal | Empfänger | Wann |
|-------|-----------|------|
| 📱 Telegram | Nur du (Admin) | Immer |
| 📧 E-Mail | Nur Kunden | Täglich 08:00 |

In [ ]:
import sys, os
sys.path.insert(0, '..')  # Projektpfad
from dotenv import load_dotenv
load_dotenv('../.env')
print('✅ .env geladen')

## 1. Konfiguration prüfen

In [ ]:
print('══ AKTIVE KANÄLE ══════════════════')
tg = bool(os.getenv('TELEGRAM_BOT_TOKEN') and os.getenv('TELEGRAM_ADMIN_CHAT_ID'))
em = os.getenv('EMAIL_ENABLED','false').lower() == 'true'
print(f'📱 Telegram (Admin): {"✅" if tg else "❌ nicht konfiguriert"}')
print(f'📧 E-Mail  (Kunden): {"✅" if em else "❌ EMAIL_ENABLED=false"}')
if em:
    print(f'   → {os.getenv("CLIENT_EMAIL","?")}')
    print(f'   → {os.getenv("CLIENT_NAME","?")}')

## 2. Telegram testen (nur du siehst das!)

In [ ]:
from src.alerts.telegram_admin import TelegramAdmin
tg = TelegramAdmin()
tg.sende('🧪 *TEST* – Data Platform v2.0 läuft!')
print('✅ Telegram-Test gesendet (nur du siehst das!)')

## 3. Kunden-E-Mail testen

In [ ]:
from src.alerts.email_kunde import EmailKunde
email = EmailKunde()

email.tagesbericht(
    umsatz={
        'heute': 45678.90, 'gestern': 42345.00,
        'veraenderung': 7.87, 'bestellungen': 123,
        'durchschnitt': 371.37, 'aktive_kunden': 89,
        'neukunden': 5, 'monat_gesamt': 987654.00,
    },
    anomalien=[
        {'datum': '2024-02-22', 'wert': 12345.00,
         'erwartet': 8900.00, 'abweichung': 38.7,
         'grund': 'Ungewöhnlich hoher Einzelauftrag', 'kritisch': False}
    ],
    prognose={
        'summe': 1450000.0, 'durchschnitt': 48333.0,
        'modell': 'Prophet', 'fehler': 4.5
    },
)
print(f'📧 E-Mail: aktiv={email.aktiv}')

## 4. Vollständiger ETL-Test

In [ ]:
from orchestrator import Orchestrator

o = Orchestrator('../config')
if o.start():
    ergebnis = o.etl()
    print('\n📋 Ergebnis:')
    for k, v in ergebnis.items():
        print(f'  {k}: {v}')
else:
    print('❌ Start fehlgeschlagen – .env prüfen')

## 5. Testdaten erstellen (wenn data/raw/ leer ist)

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

Path('../data/raw').mkdir(parents=True, exist_ok=True)

np.random.seed(42)
n = 200
kunden  = ['Müller GmbH', 'Schmidt AG', 'Weber KG', 'Fischer OHG', 'Koch GmbH']
produkte = ['Produkt A', 'Produkt B', 'Produkt C', 'Produkt D']

df = pd.DataFrame({
    'order_date': pd.date_range('2024-01-01', periods=n, freq='D').strftime('%d.%m.%Y'),
    'customer':   np.random.choice(kunden, n),
    'product':    np.random.choice(produkte, n),
    'amount':     (np.random.exponential(5000, n) + 500).round(2),
    'quantity':   np.random.randint(1, 10, n),
})

datei = '../data/raw/test_daten.csv'
df.to_csv(datei, index=False)
print(f'✅ Testdaten erstellt: {datei} ({len(df)} Zeilen)')
df.head()